Importing necessary libraries

In [181]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.express as px
from PIL import Image
import random
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from scipy import stats
import pandas as pd
from IPython.display import display, Markdown


In [182]:
root = "Poles2025"
dataset = "roadpoles_v1" 
dataset_path = Path(root) / dataset

In [183]:
split_counts = {}

for split in ["train", "valid", "test"]:
    image_dir = dataset_path / split / "images"
    images = list(image_dir.glob("*.PNG"))
    split_counts[split] = len(images)
    
# print(split_counts)
    
print(f"Training set: {split_counts["train"]} images")
print(f"Validation set: {split_counts["valid"]} images")
print(f"Testing set: {split_counts["test"]} images")



Training set: 322 images
Validation set: 92 images
Testing set: 46 images


In [184]:
negative_samples = 0
split_data = {split: {"cx": [], "cy": [], "w": [], "h": []} for split in ["train", "valid"]}
all_cx, all_cy, all_w, all_h = [], [], [], []
boxes_per_image = {"train": [], "valid": []}

for split in ["train", "valid"]:
    label_dir = dataset_path / split / "labels"
    labels = list(label_dir.glob("*.txt"))
    
    total_annotations = 0
    
    for label_file in labels:
        with open(label_file) as file:
            lines = file.readlines()
            total_annotations += len(lines)
            boxes_per_image[split].append(len(lines))
            
            if len(lines) == 0:
                negative_samples += 1
            
            for line in lines:
                _, cx, cy, w, h = line.split()
                cx, cy, w, h = float(cx), float(cy), float(w), float(h)
                split_data[split]["cx"].append(cx)
                split_data[split]["cy"].append(cy)
                split_data[split]["w"].append(w)
                split_data[split]["h"].append(h)
                all_cx.append(cx)
                all_cy.append(cy)
                all_w.append(w)
                all_h.append(h)
    
    avg = total_annotations / len(labels) if labels else 0
    print(f"{split}: {total_annotations} annotations, {avg:.2f} per image")

print(f"\nTotal bounding boxes: {len(all_cx)}")
print(f"Negative samples (no poles): {negative_samples}")

train: 392 annotations, 1.22 per image
valid: 113 annotations, 1.23 per image

Total bounding boxes: 505
Negative samples (no poles): 0


In [185]:

fig = make_subplots(rows=1, cols=2, subplot_titles=["Train", "Val"], shared_xaxes=True, shared_yaxes=True)

for i, split in enumerate(["train", "valid"], 1):
    counts = boxes_per_image[split]
    total = len(counts)
    unique_vals = sorted(set(counts))
    percentages = [100 * counts.count(v) / total for v in unique_vals]
    
    fig.add_trace(go.Bar(
        x=unique_vals,
        y=percentages,
        name=split,
        text=[f"{p:.1f}%" for p in percentages],
        textposition="outside"
    ), row=1, col=i)

fig.update_layout(title="Percentage of images with N bounding boxes (train vs val)", bargap=0.3, height=600)
fig.update_xaxes(title_text="Number of boxes", tickmode="linear", dtick=1)
fig.update_yaxes(title_text="Percentage (%)", col=1)
fig.show()

In [186]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Train", "Val"], shared_xaxes=True, shared_yaxes=True)

for i, split in enumerate(["train", "valid"], 1):
    fig.add_trace(go.Scatter(
        x=split_data[split]["w"],
        y=split_data[split]["h"],
        mode="markers",
        name=split
    ), row=1, col=i)

fig.update_layout(title="Bounding box size (w vs h)")
fig.update_xaxes(title_text="Width")
fig.update_yaxes(title_text="Height", col=1)
fig.show()

In [187]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Train", "Val"], shared_xaxes=True, shared_yaxes=True)

for i, split in enumerate(["train", "valid"], 1):
    aspect_ratio = np.array(split_data[split]["h"]) / np.array(split_data[split]["w"])
    fig.add_trace(go.Histogram(x=aspect_ratio, name=split, nbinsx=30, histnorm="percent"), row=1, col=i)

fig.update_layout(title="Aspect ratio (h/w) per split", bargap=0.1)
fig.update_xaxes(title_text="Height/Width")
fig.update_yaxes(title_text="Percentage (%)", col=1)
fig.show()

In [188]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Train", "Val"])

for i, split in enumerate(["train", "valid"], 1):
    fig.add_trace(go.Histogram2d(
        x=split_data[split]["cx"],
        y=split_data[split]["cy"],
        colorscale="Viridis",
        histnorm="percent",
        showscale=(i == 2),
        name=split
    ), row=1, col=i)

fig.update_layout(title="Heatmap of bounding box center point (cx vs cy)")
fig.update_xaxes(title_text="cx")
fig.update_yaxes(title_text="cy", col=1)
fig.show()

In [189]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Train", "Val"], shared_xaxes=True, shared_yaxes=True)

for i, split in enumerate(["train", "valid"], 1):
    rel_size = np.array(split_data[split]["w"]) * np.array(split_data[split]["h"])
    fig.add_trace(go.Histogram(x=rel_size, name=split, nbinsx=30, histnorm="percent"), row=1, col=i)

fig.update_layout(title="Relative size of boxes (w×h) per split", bargap=0.1)
fig.update_xaxes(title_text="Width × Height")
fig.update_yaxes(title_text="Percentage (%)", col=1)
fig.show()

In [190]:
image_paths_brightness = []
widths, heights, brightness, brightness_splits = [], [], [], []
weights = np.array([0.299, 0.587, 0.114])

for split in ["train", "valid", "test"]:
    image_dir = dataset_path / split / "images"
    images = list(image_dir.glob("*.PNG"))
    
    for image_path in images:
        with Image.open(image_path) as image:
            image_paths_brightness.append(image_path)
            widths.append(image.width)
            heights.append(image.height)
            image_array = np.array(image)
            brightness.append(np.mean(image_array @ weights))
            brightness_splits.append(split)
            
check_widths = len(set(widths)) == 1
check_heights = len(set(heights)) == 1

if check_widths and check_heights:
    print(f"All images have the same resolution: {widths[0]}x{heights[0]}")
elif check_widths:
    print(f"All images have the same width: {widths[0]}px, but heights vary")
elif check_heights:
    print(f"All images have the same height: {heights[0]}px, but widths vary")
else:
    print("Images have varying resolutions")

All images have the same resolution: 1920x1208


In [191]:
fig = make_subplots(rows=1, cols=3, subplot_titles=["Train", "Val", "Test"], shared_xaxes=True, shared_yaxes=True)

for i, split in enumerate(["train", "valid", "test"], 1):
    split_brightness = [b for b, s in zip(brightness, brightness_splits) if s == split]
    fig.add_trace(go.Histogram(x=split_brightness, name=split, nbinsx=30, histnorm="percent"), row=1, col=i)

fig.update_layout(title="Average brightness per image (0-255)", bargap=0.1)
fig.update_xaxes(title_text="Brightness")
fig.update_yaxes(title_text="Percentage (%)", col=1)
fig.show()

In [192]:
rows = []
for split in ["train", "valid"]:
    w = np.array(split_data[split]["w"])
    h = np.array(split_data[split]["h"])
    cx = np.array(split_data[split]["cx"])
    cy = np.array(split_data[split]["cy"])
    ar = h / w
    rel = w * h
    bpi = np.array(boxes_per_image[split])
    bright = np.array([b for b, s in zip(brightness, brightness_splits) if s == split])

    for name, arr in [("w", w), ("h", h), ("cx", cx), ("cy", cy), ("aspect ratio", ar), ("w×h", rel), ("boxes/image", bpi), ("avg brightness", bright)]:
        rows.append({
            "Variable": name,
            "Split": split,
            "Mean": round(float(np.mean(arr)), 4),
            "Std": round(float(np.std(arr)), 4),
        })

test_bright = np.array([b for b, s in zip(brightness, brightness_splits) if s == "test"])
rows.append({
    "Variable": "avg brightness",
    "Split": "test",
    "Mean": round(float(np.mean(test_bright)), 4),
    "Std": round(float(np.std(test_bright)), 4),
})

df = pd.DataFrame(rows)
pivot = df.pivot(index="Variable", columns="Split", values=["Mean", "Std"])
pivot.columns = [f"{stat} ({split})" for stat, split in pivot.columns]
pivot = pivot[["Mean (train)", "Mean (valid)", "Mean (test)", "Std (train)", "Std (valid)", "Std (test)"]]

display(Markdown("## Summary Statistics (train vs val vs test)"))
display(pivot.round(4))

## Summary Statistics (train vs val vs test)

,Mean (train),Mean (valid),Mean (test),Std (train),Std (valid),Std (test)
Variable,,,,,,
aspect ratio,16.3222,16.6971,NaN,7.5897,7.3429,NaN
avg brightness,94.2242,92.8531,93.2093,7.8651,5.6434,4.1266
boxes/image,1.2174,1.2283,NaN,0.4273,0.4449,NaN
cx,0.4609,0.4556,NaN,0.2467,0.2369,NaN
cy,0.6073,0.6053,NaN,0.0325,0.0314,NaN
h,0.1219,0.1124,NaN,0.0728,0.0714,NaN
w,0.0091,0.0081,NaN,0.0086,0.0085,NaN
w×h,0.0015,0.0013,NaN,0.0029,0.0034,NaN


In [193]:
variables = ["w", "h", "cx", "cy", "aspect ratio", "w×h", "boxes/image", "avg brightness"]
colors = {"train": "#636EFA", "valid": "#EF553B", "test": "#00CC96"}

cols = 4
rows = 2
fig = make_subplots(rows=rows, cols=cols, subplot_titles=variables)

for i, var in enumerate(variables):
    row = i // cols + 1
    col = i % cols + 1
    splits = ["train", "valid", "test"] if var == "avg brightness" else ["train", "valid"]
    
    for split in splits:
        fig.add_trace(go.Bar(
            name=split,
            x=[split],
            y=[pivot.loc[var][f"Mean ({split})"]],
            error_y=dict(type="data", array=[pivot.loc[var][f"Std ({split})"]], visible=True),
            marker_color=colors[split],
            showlegend=(i == 0),
        ), row=row, col=col)

fig.update_layout(
    title="Mean ± Std per variable",
    height=500,
    barmode="group",
    legend_title="Split",
)

fig.add_trace(go.Bar(
    name="test",
    x=[None],
    y=[None],
    marker_color=colors["test"],
    showlegend=True,
    legendgroup="test",
))

fig.show()